In [ ]:
from hdps.skymap import TanSegmentActivationManager

#"""AICBase (coming soon) - Builds a catalog of ''always-on'' active minipix from AIC Refcat [Identifies unique minipix associated with AICRefcat sources?  Activates all unique minipix?]"""


testing for building AIC Base


- AIC Base
    - minipix catalog of every "always-on" minipix that contains a source(s) from AIC Refcat (and adjacent minipix needed to save background annuli?)
    - built with `AICBase` class




*What needs to happen:*

- AIC Base (unique minipix that contain all AIC Refcat sources)
    - "build" AIC Base means get all the minipix that contain sources from AIC Refcat
        - input should be:
            - AIC Refcat (like the path to where it is saved)
            - an on-disk minipix path for the TanSeg Activation Manager
            - how much background annulus you want to save around each point source
            - how to deal with extended objects.
        - output should be:
            - a list of minipix to activate (containing both sources and the adjacent minipix needed to save enough background around each source)
            - possibly also a collection of active minipix metadata (saved in the minipix path stipulated in inputs for the TanSeg Activation Manager)
            - ***minipix are activated here too?***
    - Build Steps:
        - select all unique minipix/tanseg IDs in AIC Refcat
        - ***activate those minipix?***
        - possibly maintain a list of currently active minipix

In [ ]:
# Get unique minipix.  Since minipixp indices are not globally unique, you must consider the tanseg id.
# this is now sitting in the get_minipix() function
#unique_minipix = np.unique(atlas_single_part_minipix[['tanseg_id','local_idx']], axis=0)
#print("I calculate ",len(unique_minipix)," unique minipix in atlas_single_part")


### <font color='#f1c232'>...Initialize catalog</font>

This package should have the functionality to activate the minipix that refcat sources are in ("initialize a catalog" is what it's called in the current claude-generated HDPS documentation).

This pertains to AICBase.

**What needs to happen:**

The unique tanseg ID & minipix index pairs must be retrieved from AIC Refcat, and initialized.  I first thought that this could not be done with map_partitions + compute, but seemingly it works now?  

Old: ~~While the map_partition() function works for looking up minipix/tile IDs, it does not work for something like activating minipix for all the sources in a dataframe, as the input would be a dataframe but the act of writing minipix files to disk cannot be handled in a "lazy" manner.  This is why I need to use __[CatalogStream](https://docs.lsdb.io/en/latest/reference/api/lsdb.streams.CatalogStream.html#lsdb.streams.CatalogStream)__ to loop through each partition, load the data to get the minipix/tile IDS, and activate them.  Sure it takes a long time, but it needs to be done, I think, unless I can simply just call in the minipix/tile IDs and not the rest of the data....... that would take a little less time?  or be a little less memory-intensive?~~


~~**The issue I'm having:***~~

~~When I load a catalog lazily with LSDB, I impose certain filters in order to select appropriate spatial and magnitude ranges.~~

~~The problem is that if you load a catalog with filters, CatalogStream doesn't recognize columns you added after opening the cat.  We do indeed need to add columns after opening the refcat if tanseg_id & local_idx columns are not included.~~


<font color="#e69138">First, define a function that accepts a dataframe with minipix index and tanseg ID columns, and returns a dataframe with just 2 columns: all unique minipix/tanseg IDs (read in AIC Refcat and generate AICBase).  Also define a function that accepts a this 2-col dataframe and activates all the minipix. </font>

In [ ]:
# Define a function to activate minipix (initialization)
# This function will show that you cannot successfuly use map_partitions() to activate minipix.  
# This function therefore should accept a dataframe as first argument (requirement of map_partitions()),
# a TanSeg Activation Manager for initialization, and the df's column names for tanseg IDs & minipix indices.

def initialize_refcat(df,manager,tile_col="tanseg_id",minipix_col="local_idx",verbose=False):
    """
    Initialize active minipix corresponding to sources in reference catalog
    
    If verbose, uses function initialize_from_source_catalog() from hdps_skymap; the output is Very verbose.

    If not verbose, takes same code from hdps_skymap and removes the default verbose output.  For Katie's sanity.

    Parameters
    ----------
    df (DataFrame)
    manager (TansegActivationManager)
    tile_col (str, default="tanseg_id")
    minipix_col (str, default="local_idx")
    verbose (bool, default=False) - verbose or not?

    Returns
    -------
    df # total_active_minipixes (int) number of active unique minipix 

    """
    if tile_col not in df.columns or minipix_col not in df.columns:
        print("Tile/minipix columns not in dataframe, goodbye")
        return 0

    # make a df that is only unique combinations of tile_col,minipix_col
    #df = df[[tile_col,minipix_col]].drop_duplicates()


    if verbose:
        manager.initialize_from_source_catalog(
            df,
            tanseg_column=tile_col,
            local_idx_column=minipix_col,
            default_duration=100 * u.year,
        )
        #total_active_minipixes = len(df[[tile_col,minipix_col]])#.drop_duplicates())

    else:
        default_duration=100 * u.year
        tanseg_ids = df[tile_col].unique()
        #total_segments = 0
        #total_active_minipixes = 0

        for tanseg_id in tanseg_ids:
            # Get unique local indices for this segment
            segment_rows = df[df[tile_col] == tanseg_id]
            active_local_indices = set(segment_rows[minipix_col].unique())

            manager.initialize_segment_metadata(
                int(tanseg_id),
                active_minipixes=active_local_indices,
                default_duration=default_duration,
            )

            #total_segments += 1
            #total_active_minipixes += len(active_local_indices)

    return df#total_active_minipixes

In [ ]:
# Show how CatalogStream works with a toy catalog
cat = lsdb.generate_catalog(500, 10, seed=1)
print(cat.npartitions," partitions in toy cat")

cat_stream = CatalogStream(catalog=cat, partitions_per_chunk=4, seed=1)
for chunk in cat_stream:
    print(len(chunk)," rows in chunk") #print(chunk[['a','b']].drop_duplicates())

<font color="#e69138">First, show that you can initialize minipix if given a fully-loaded dataframe, but not using map_partitions() on a lazily-loaded dataframe, until you compute() the dataframe after.</font>

ordr = 3     # healpix order of partition you want (I looked this pair up in my notebook for testing mapping functions to partitions with LSDB)
pixid = 368  # ID of partition you want
atlas_pix_lookup = lsdb.open_catalog(atlas_path,search_filter=lsdb.PixelSearch((ordr,pixid)),filters=atlas_filters)
atlas_single_part = atlas_pix_lookup.compute()

In [ ]:
# Show that this works for a dataframe that has all data loaded:
print("Show that initialize_refcat() works on a fully-loaded dataframe (send to ./data).....")

atlas_single_part = get_minipix(atlas_single_part,lookup,"ra","dec")
atlas_single_part_initialized = initialize_refcat(atlas_single_part,manager,"tanseg_id","local_idx",quiet=False)

In [ ]:
# Show that map_partitions with the function that initializes minipix doesnt't work, since the catalog is lazily loaded.
print("Show that applying map_partitions(initialize_refcat) to a lazily-loaded catalog doesn't initialize the minipix:")
#atlas_cone = lsdb.open_catalog(atlas_path,search_filter=lsdb.ConeSearch(ra=40, dec=30, radius_arcsec=5000),filters=atlas_filters)
#atlas_cone = atlas_cone.map_partitions(get_minipix,lookup,"ra","dec") # get minipix
atlas_pix_lookup = atlas_pix_lookup.map_partitions(get_minipix,lookup,"ra","dec")
#atlas_pix_lookup_initialized = atlas_pix_lookup.map_partitions(initialize_refcat,test_manager,"tanseg_id","local_idx",quiet=False)
atlas_pix_lookup = atlas_pix_lookup.map_partitions(initialize_refcat,test_manager,"tanseg_id","local_idx",quiet=False)

# computing the catalog used to return an error, now it just works:
print("\ncompute:")

with Client(n_workers=4, memory_limit="auto", threads_per_worker=1,processes=False) as client:
    atlas_pix_lookup_initialized_data = atlas_pix_lookup.compute()

In [ ]:
# Check initialized segments & activated minipix from both methods to make sure all minipix are correctly activated
initialized_segments = manager.list_initialized_segments()
tansegs = []
minipixes = [] 
for tseg in initialized_segments:
    tseg_minipix = list(manager.get_active_minipixes(tseg).keys())
    tansegs = tansegs + list(range(len(tseg_minipix)))
    minipixes = minipixes + tseg_minipix

atlas_single_part_minipix = pd.DataFrame({'tanseg_id': tansegs, 'local_idx': minipixes})
print("Initializing with a fully-loaded dataframe:")
print(len(atlas_single_part_minipix)," minipix activated")
atlas_single_part_minipix.drop_duplicates()
print(len(atlas_single_part_minipix)," minipix activated, no duplicates")

In [ ]:
# Check initialized segments & activated minipix from both methods to make sure all minipix are correctly activated
initialized_segments = test_manager.list_initialized_segments()
tansegs = []
minipixes = [] 
for tseg in initialized_segments:
    tseg_minipix = list(test_manager.get_active_minipixes(tseg).keys())
    tansegs = tansegs + list(range(len(tseg_minipix)))
    minipixes = minipixes + tseg_minipix

atlas_pix_lookup_minipix = pd.DataFrame({'tanseg_id': tansegs, 'local_idx': minipixes})
print("Initializing with map_partitions() on a lazily-loaded dataframe:")
print(len(atlas_pix_lookup_minipix)," minipix activated")
atlas_pix_lookup_minipix.drop_duplicates()
print(len(atlas_pix_lookup_minipix)," minipix activated, no duplicates")

<font color="#e69138">It seems that initializing the refcat with map_partitions(initialize_refcat) activates a different number of minipix each time I try, never the same amount that you get if you initialize on a fully-loaded dataframe.</font>

Granted, I am running map_partitions() on a lazily-loaded catalog twice - once to lookup minipixes, then again to initialize them.  This could be an issue; in the fully-realized functionality, I will be reading in a catalog that already has a minipix column.  I will try that out.  

In [ ]:
tansegs_data = !ls -ltrh data/aic_base_minipix
tansegs_data = set([i.split(" ")[-1] for i in tansegs_data[1:]])

tansegs_test = !ls -ltrh test_minipix/data/aic_base_minipix
tansegs_test = set([i.split(" ")[-1] for i in tansegs_test[1:]])

print(list(tansegs_data-tansegs_test))

In [ ]:
# Instead of mapping the initialization function to the lazily loaded catalog, you must stream the catalog
# in to apply the function to however many chunks at once.


# see how many pixels/partitions
pixels = atlas_frac.get_healpix_pixels()
print(atlas_frac.npartitions," partitions in ATLAS")
print(len(pixels))#/2000)

In [ ]:

# takes 21 or 13 minutes to initialize 0.1% of ATLAS-Refcat2 with 200 partitions per chunk (~110 chunks).
# but if you filter on mag<20.5 & dec>-20, it takes 10 minutes with 200 partions per chunk.
print("Show that using CatalogStream can go through a catalog partition-by-partition, and this way the function that initializes a refcat can be applied.")

cat_stream = CatalogStream(catalog=atlas_frac['tanseg_id','local_idx',"g","dec"], partitions_per_chunk=10, seed=1)
i=1
for chunk in cat_stream:
    chunk = chunk[(chunk['g']>20.5) & (chunk['dec']>-20)]
    minipix = initialize_refcat_quiet_test(chunk,manager)
    print("initialized chunk ",i," length ",len(chunk)," with ",minipix," minipix")
    i+=1
    #print(len(chunk),len(chunk[['tanseg_id','local_idx']].drop_duplicates()))

### <font color='#f1c232'>...Initialize cat in chunks - are all minipix activated only once?</font>
Checking that if you initialize part of a catalog with a manager that takes a minipix path, then initialize another part of that catalog, both sets of minipix are kept in that manager's minipix path on disk.

In [ ]:
# lookup 2 partitions of ATLAS
atlas_pix_lookup = lsdb.open_catalog(atlas_path,search_filter=lsdb.PixelSearch((3,368))) #PixelSearch((order,pixelid))
atlas_part_1 = atlas_pix_lookup.compute()
print(f"Single partition of HATSed 0.1% of ATLAS lazily and fully loaded ({len(atlas_part_1)}) rows)")


atlas_pix_lookup = lsdb.open_catalog(atlas_path,search_filter=lsdb.PixelSearch((7,165870))) #PixelSearch((order,pixelid))
atlas_part_2 = atlas_pix_lookup.compute()
print(f"Single partition of HATSed 0.1% of ATLAS lazily and fully loaded ({len(atlas_part_2)}) rows)")



In [ ]:
# initialize one partition
atlas_1_ids = lookup_minipix_idx(atlas_part_1,ra_col="ra",dec_col="dec")
initialize_refcat_minipix(atlas_1_ids,manager_test)


In [ ]:
# initialize the other partition with the same manager as above:
atlas_2_ids = lookup_minipix_idx(atlas_part_2,ra_col="ra",dec_col="dec")
initialize_refcat_minipix(atlas_2_ids,manager_test)

In [ ]:
#413288  active minipix in  94134  tiles for  431545  sources, when applying minipix function directly to chunk, filtering catalog at opening, 14 minutes at 200partitions/chunk, 0.001 atlas
#413288  active minipix in  94134  tiles for  431545  sources, when applying minipix function with map_partitions and filtering catalog in CatalogStream, 11 minutes 


Now, if you open a terminal after initializing each partition, you can do an $ls in the minipix path and see that both sets of activated minipix are indeed stored.

<font color="#e69138">AICBase rewrite</font>

AICBase is a class that should read in AIC Refcat, create a list unique of "always-on" minipix to activate, save that list somewher(?), and (if requested) activate those minipix.  It thus creates AIC Base.  

In [ ]:
class AICBase:
    """
    Builds AIC Base, consisting of all "always-on" active minipix 
    that are pulled from AIC Refcat. 
    Initializes 

    Parameters
    ----------
    aic_refcat : AICRefcat object 
        Reference source catalog to use as AIC Refcat
    minipix_path : str
        Path to directory in which to store minipix metadata
    n_tiles : int
        Number of occupied tiles
    n_minipix : int
        Number of unique occupied minipix


    """

    def __init__(
        self,
        aic_refcat = None,
        minipix_path = "/net/scratch/kmfas/data/aic_base_minipix/",
        n_tiles = None,
        n_minipix = None
    ):

        self.aic_refcat = aic_refcat
        self.minipix_path = minipix_path 
        self.n_tiles = n_tiles
        self.n_minipix = n_minipix

        # Derived parameters
        from hdps.skymap import SegmentationScheme
        self.tile_arcsec = SegmentationScheme.DEFAULT_TILE_SIZE_ARCSEC
        self.overlap_arcsec = SegmentationScheme.DEFAULT_OVERLAP_ARCSEC
        self.pixel_scale_arcsec = SegmentationScheme.DEFAULT_PIXEL_SCALE

        #self.scheme = SegmentationScheme.allsky()
        self._manager = None


    @property
    def manager(self):
        """
        TanSegment Manager activating minipix.
        The manager is created lazily on first access and cached for reuse.

        Returns
        --------
        self._manager (TanSegmentActivationManager) Tanseg Activation manager to store minipix data in self.minipix_path 
        """

        # if no manager, create one
        if self._manager is None:
            from hdps.skymap import TanSegmentActivationManager
            self._manager = TanSegmentActivationManager(self.minipix_path)

        return self._manager


    def initialize_minipix(self,quiet=True):
        """
        Initialize active minipix corresponding to sources in dframe,
        * Can't be used with map_partitions()

        Parameters
        ----------
        quiet (bool) Don't print out which tanseg/minipix have been activated.  For Katie's sanity.
                        Default = True

        Returns
        -------
        total_segments (int) Total number of unique tansegments
        total_active_minipixes (int) Total number of active minipix
        """

        # setup
        default_duration=100 * u.year # for how long are the minipixes active
        dframe = dframe[[self.tile_col,self.minipix_col]].drop_duplicates() # only want tanseg & minipix columns, and only unique pairs of tanseg+minipix 
        tanseg_ids = dframe[self.tile_col].unique() # all the unique tansegs 

        # if not quiet, use initialize_from_source_catalog()
        if not quiet:

            # initialize
            self.manager.initialize_from_source_catalog(
                dframe,
                tanseg_column=self.tile_col,
                local_idx_column=self.minipix_col,
                default_duration=default_duration,
            )

            total_segments = len(tanseg_ids)
            total_active_minipixes = len(dframe)

        # if quiet, use initialize_segment_metadata() and loop through unique tansegs 
        else:

            total_segments = 0
            total_active_minipixes = 0

            # loop through tansegs, get minipix for each and activate!  (this is what happens in initialize_source_catalog(), just without printout)
            for tanseg_id in tanseg_ids:

                # Get unique local indices for this segment
                segment_rows = dframe[dframe[self.tile_col] == tanseg_id]
                active_local_indices = set(segment_rows[self.minipix_col].unique())

                # initialize
                self.manager.initialize_segment_metadata(
                    int(tanseg_id),
                    active_minipixes=active_local_indices,
                    default_duration=default_duration,
                )

                total_segments += 1
                total_active_minipixes += len(active_local_indices)
        
        return total_segments,total_active_minipixes


    def initialize_catalog(cls,smart=True,verbose=False):
        """
        Initialize AIC Base (call refcat & activate corresponding minipix)

        Parameters
        ----------
        smart (bool) Smart-ly select number of reference catalog partitions to initialize at once (get refcat stats, takes longer)
                        Default = True
        verbose (bool) Whether to print out information
                        Default = False
        Returns
        -------

        """
        #refcat,npartitions,nsrc_in_largest_partition = aic_refcat

        if smart:
            max_src_per_chunk = 2000000 # max sources per chunk
            parts_per_chunk = min(200,max_src_per_chunk//aic_refcat.nsrc_largest) # number of partitions per chunk, don't allow more than than 200 (this is basically to keep the number of sources per chunk under 2million)
        else: 
            parts_per_chunk = 2 # a safe amount, in case there's like 2million sources per partition (tryin to protect your memory here)

        # using CatalogStream, initialize all minipix
        cat_stream = CatalogStream(catalog=cls.aic_refcat.catalog, partitions_per_chunk=parts_per_chunk, seed=1)
        tot_tiles = 0
        tot_minipix_active = 0
        nsources = 0
        if verbose: print("Initializing refcat minipix, with ",parts_per_chunk," partitions per chunk\n***Will take 10-40 minutes, beware!")
        for chunk in cat_stream:
            chunk = chunk[(chunk[cls.mag_col]<=cls.mag_lim) & (chunk[self.ra_col]>=self.ra_min) & (chunk[self.ra_col]<=self.ra_max) & (chunk[self.dec_col]>=self.dec_min) & (chunk[self.dec_col]<=self.dec_max)]
            segments,minipix = self.initialize_minipix(chunk)
            nsrc_in_chunk = len(chunk)
            if verbose: print("    initialized ",minipix," minipix for ",nsrc_in_chunk," sources")
            tot_minipix_active = tot_minipix_active+minipix
            tot_tiles = tot_tiles+segments
            nsources = nsources+nsrc_in_chunk
        if verbose: print("Initialized ",tot_minipix_active," active minipix in ",tot_tiles," tiles for ",nsources," sources in reference catalog")

        ######################
        # ******************** do we want to have a list of all unique active tanseg+minipix?  If so, should it be saved to disk or kept in memory?  Important questions.....
        ######################

        return cls(tot_tiles,tot_minipix_active)